# Xenium Run 2 – 02 Cell-type and Tg-subtype annotation

**Input:** `20251009_Run2_reSeg_QC.h5ad` (notebook 01)
**Output:**
* `20261019_ALLcells_run2.h5ad` – all annotated cells (`compartment_annotation_2`) → cell-type composition
* `20260520_TBcells_run2.h5ad` – Tg + T + B cells (`Tg_annotation_full`) → T/B-border inference

Same marker sets, thresholds and two-step Tg-subtype annotation as run 1
(`Xenium_Run1_02_annotation.ipynb`).

> **Difference to run 1 (as run for the manuscript):** Tg T-cell candidates are taken from the
> first annotation of notebook 01 (`compartment_annotation`, highest module score), whereas run 1 uses
> `compartment_annotation_2` (highest score **and** ≥ 33rd percentile). The subsequent Tg QC and the
> top-25 % Tg-score filter are identical.

Next notebook: `Xenium_03_TB_border_combined.ipynb`

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats, sparse

In [ ]:
# ------------------------------------------------------------
# Paths relative to this folder (adapt BASE_DIR if your data live elsewhere)
# ------------------------------------------------------------
BASE_DIR = ".."
RAW_DIR  = f"{BASE_DIR}/01_Raw"
DATA_DIR = f"{BASE_DIR}/02_ProcessedData"

QC_IN   = f"{DATA_DIR}/20251009_Run2_reSeg_QC.h5ad"   # from notebook 01
ALL_OUT = f"{DATA_DIR}/20261019_ALLcells_run2.h5ad"   # all annotated cells
TB_OUT  = f"{DATA_DIR}/20260520_TBcells_run2.h5ad"    # Tg + T + B cells

## 2. Load data

In [ ]:
adata = sc.read(QC_IN)
adata

Colours, sample order and marker genes for the major cell types.

In [ ]:
ln_major_colors = {
    "B_cells": "#4A90E2",        # soft blue
    "T_cells": "#50E3C2",        # gentle green
    "Tg_T_cells": "#D0021B",     # magenta / highlight
    "Macrophages": "#F5A623",    # warm gold
    "Stromal_cells": "#9B9B9B",  # neutral gray-violet
    "Lymphatic_EC": "#2E8B57",    # teal-cyan,
    "Tregs": "#7B6FD0",          # muted violet
    "cDC1": "#084594",           # deep navy blue (clearly blue)
    "cDC2": "#6B8E23",           # olive / yellow-green (clearly green)
    "GC_B_cells": "#6A3D9A" ,     # deep purple
     "Neutrophils": "#636363",
    "NA": "white"
}
ordered_timepoints = [
   '16:Day0_0383',
    '15:Day2_0386',
    '13:Day3_0390',
    '14:Day3_0390',
    '12:Day4_0393',
    '11:Day6_0396',
    'Unknown'
]

In [ ]:
ln_major_markers = {
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"]
}

### Composition with the first annotation (notebook 01)

In [ ]:
# Remove entries with "Unknown" timepoint
adata_filtered = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata_filtered.obs.groupby(["timepoint", "compartment_annotation"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)
cell_fractions

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Major cell types – module score with per-type threshold

Highest module score, only if ≥ 33rd percentile of that score; otherwise `NA`. `Unknown` cells removed.

In [ ]:
# 1. Compute gene set scores
for comp, genes in ln_major_markers.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{comp}_score")

# 2. Define score columns
compartments = list(ln_major_markers.keys())
score_cols = [f"{c}_score" for c in compartments]
scores = adata.obs[score_cols].copy()

# 3. Per-cell type thresholds: top 25% of cells retained
thresholds = scores.quantile(0.33)

# 4. Best label and score per cell
adata.obs["best_compartment_score_col"] = scores.idxmax(axis=1)
adata.obs["best_compartment_score"] = scores.max(axis=1)

# 5. Map threshold of the winning compartment to each cell
adata.obs["best_compartment_threshold"] = adata.obs["best_compartment_score_col"].map(thresholds)

# 6. Final annotation
adata.obs["compartment_annotation_2"] = np.where(
    adata.obs["best_compartment_score"] >= adata.obs["best_compartment_threshold"],
    adata.obs["best_compartment_score_col"].str.replace("_score", "", regex=False),
    pd.NA
)

In [ ]:
# Remove entries with "Unknown" timepoint
adata = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata.obs.groupby(["timepoint", "compartment_annotation_2"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)
cell_fractions

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation_2', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Tg T cells

### 4.1 QC (median ± 3 MAD for counts and genes, Tukey fences for area, 2D-doublet filter)

In [ ]:
adata_Tg_Tcells = adata[adata.obs['compartment_annotation'] == 'Tg_T_cells'].copy()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata_Tg_Tcells.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata_Tg_Tcells.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata_Tg_Tcells.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata_Tg_Tcells.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
# Helper: robust bounds using MAD
def mad_bounds(x, k_low=3, k_high=3, log=False):
    x = np.asarray(x, dtype=float)
    x = np.log1p(x) if log else x
    med = np.nanmedian(x)
    mad = stats.median_abs_deviation(x, nan_policy='omit', scale='normal')
    lower = med - k_low * mad
    upper = med + k_high * mad
    return (np.expm1(lower) if log else lower,
            np.expm1(upper) if log else upper)

# 2) counts & genes: drop low and high outliers
lc_lo, lc_hi = mad_bounds(adata_Tg_Tcells.obs['total_counts'], 3, 3, log=True)
lg_lo, lg_hi = mad_bounds(adata_Tg_Tcells.obs['n_genes_by_counts'], 3, 3, log=False)

m_counts = (adata_Tg_Tcells.obs['total_counts'] >= lc_lo) & (adata_Tg_Tcells.obs['total_counts'] <= lc_hi)
m_genes  = (adata_Tg_Tcells.obs['n_genes_by_counts'] >= lg_lo) & (adata_Tg_Tcells.obs['n_genes_by_counts'] <= lg_hi)

# 3) cell area: Tukey bounds
q1, q3 = np.nanpercentile(adata_Tg_Tcells.obs['cell_area'], [25, 75])
iqr = q3 - q1
area_lo = q1 - 1.5*iqr
area_hi = q3 + 1.5*iqr
m_area = (adata_Tg_Tcells.obs['cell_area'] >= area_lo) & (adata_Tg_Tcells.obs['cell_area'] <= area_hi)

# 4) nucleus ratio: keep properly segmented cells
#nr = adata_Tg_Tcells.obs['nucleus_ratio']
#m_nuc = (nr >= 0.30) & (nr <= 0.90) & (~np.isclose(nr, 1.0))

# Optional extra guard against doublets: very large area & very high counts simultaneously
p99_counts = np.nanpercentile(adata_Tg_Tcells.obs['total_counts'], 99)
p99_area   = np.nanpercentile(adata_Tg_Tcells.obs['cell_area'], 99)
m_not_doublet2d = ~((adata_Tg_Tcells.obs['total_counts'] >= p99_counts) & (adata_Tg_Tcells.obs['cell_area'] >= p99_area))

# Combine all filters
mask = m_counts & m_genes & m_area & m_not_doublet2d
adata_tg_filt = adata_Tg_Tcells[mask].copy()

print(adata_Tg_Tcells.shape, "→", adata_tg_filt.shape)

### 4.2 Keep Tg T cells with a high Tg module score (top 25 %)

In [ ]:
adata_tg_filt.obs["Tg_T_cells_score"].describe()

In [ ]:
threshold = adata_tg_filt.obs["Tg_T_cells_score"].quantile(0.75)  # keep top 25 %
adata_Tg_Tcells_filt = adata_tg_filt[
    adata_tg_filt.obs["Tg_T_cells_score"] > threshold
].copy()

print(f"Filtered from {adata_tg_filt.n_obs} to {adata_Tg_Tcells_filt.n_obs} cells.")

### 4.3 Tg subtypes – two-step annotation (identical to run 1)

In [ ]:
# ------------------------------------------------------------
# 1. Define marker panels
# ------------------------------------------------------------

# Lineage-defining markers
tg_lineage_markers = {
    "Tfh_lineage": [
        "Cxcr5",
        "Pdcd1",
        "Il21",
        "Bcl6",
        "Icos"
    ],
    "Th1_lineage": [
        "Cxcr3",
        "Tbx21",
        "S100a4"
    ]
}

# Maturation markers within each lineage
tg_maturation_markers = {
    "Tfh_early": [
        "Zbtb20",
        "Arid5b",
        "Tcf7",
        "Ccr7",
        "Bach2"
    ],
    "Tfh_mature": [
        "Cxcr5",
        "Pdcd1",
        "Il21",
        "Bcl6"
    ],

    "Th1_early": [
        "Lyar",
        "Fabp5",
        "Cenpw",
        "Il2ra"
    ],
    "Th1_mature": [
        "Cxcr3",
        "Tbx21",
        "S100a4"
    ]
}


# ------------------------------------------------------------
# 2. Compute lineage scores
# ------------------------------------------------------------

for lineage, genes in tg_lineage_markers.items():

    genes_present = [
        g for g in genes
        if g in adata_Tg_Tcells_filt.var_names
    ]

    sc.tl.score_genes(
        adata_Tg_Tcells_filt,
        gene_list=genes_present,
        score_name=f"{lineage}_score"
    )


# ------------------------------------------------------------
# 3. Classify Tfh-like vs Th1-like
# ------------------------------------------------------------

lineage_score_cols = [
    "Tfh_lineage_score",
    "Th1_lineage_score"
]

adata_Tg_Tcells_filt.obs["Tg_lineage"] = (
    adata_Tg_Tcells_filt.obs[lineage_score_cols]
    .idxmax(axis=1)
    .str.replace("_lineage_score", "", regex=False)
)

# Rename to nicer labels
adata_Tg_Tcells_filt.obs["Tg_lineage"] = (
    adata_Tg_Tcells_filt.obs["Tg_lineage"]
    .replace({
        "Tfh": "Tfh",
        "Th1": "Th1"
    })
)


# ------------------------------------------------------------
# 4. Calculate lineage margin
# ------------------------------------------------------------
# Useful for identifying ambiguous cells

adata_Tg_Tcells_filt.obs["Tg_lineage_margin"] = (
    adata_Tg_Tcells_filt.obs["Tfh_lineage_score"]
    - adata_Tg_Tcells_filt.obs["Th1_lineage_score"]
)

# Absolute margin = confidence in lineage assignment
adata_Tg_Tcells_filt.obs["Tg_lineage_confidence"] = (
    adata_Tg_Tcells_filt.obs["Tg_lineage_margin"].abs()
)


# ------------------------------------------------------------
# 5. Compute maturation scores
# ------------------------------------------------------------

for state, genes in tg_maturation_markers.items():

    genes_present = [
        g for g in genes
        if g in adata_Tg_Tcells_filt.var_names
    ]

    sc.tl.score_genes(
        adata_Tg_Tcells_filt,
        gene_list=genes_present,
        score_name=f"{state}_score"
    )


# ------------------------------------------------------------
# 6. Calculate continuous maturation scores
# ------------------------------------------------------------

# Positive = more mature Tfh-like
# Negative = more early/pre-Tfh-like
adata_Tg_Tcells_filt.obs["Tfh_maturation_score"] = (
    adata_Tg_Tcells_filt.obs["Tfh_mature_score"]
    - adata_Tg_Tcells_filt.obs["Tfh_early_score"]
)

# Positive = more mature Th1-like
# Negative = more early/pre-Th1-like
adata_Tg_Tcells_filt.obs["Th1_maturation_score"] = (
    adata_Tg_Tcells_filt.obs["Th1_mature_score"]
    - adata_Tg_Tcells_filt.obs["Th1_early_score"]
)


# ------------------------------------------------------------
# 7. Convert maturation score into 4 categorical states
# ------------------------------------------------------------

def assign_tg_state(row):

    if row["Tg_lineage"] == "Tfh":

        if row["Tfh_maturation_score"] > 0:
            return "Tg_Tfh"
        else:
            return "Tg_pre_Tfh"

    elif row["Tg_lineage"] == "Th1":

        if row["Th1_maturation_score"] > 0:
            return "Tg_Th1"
        else:
            return "Tg_pre_Th1"

    return "Unassigned"


adata_Tg_Tcells_filt.obs["Tg_annotation"] = (
    adata_Tg_Tcells_filt.obs.apply(
        assign_tg_state,
        axis=1
    )
)


# ------------------------------------------------------------
# 8. Set category order
# ------------------------------------------------------------

tg_order = [
    "Tg_pre_Tfh",
    "Tg_Tfh",
    "Tg_pre_Th1",
    "Tg_Th1"
]

adata_Tg_Tcells_filt.obs["Tg_annotation"] = pd.Categorical(
    adata_Tg_Tcells_filt.obs["Tg_annotation"],
    categories=tg_order,
    ordered=True
)


# ------------------------------------------------------------
# 9. Extract day from timepoint
# ------------------------------------------------------------

# Example:
# "1:Day6_0394" -> "Day6"

adata_Tg_Tcells_filt.obs["day"] = (
    adata_Tg_Tcells_filt.obs["timepoint"]
    .astype(str)
    .str.extract(r"(Day\d+)")[0]
)

day_order = [
    "Day0",
    "Day2",
    "Day3",
    "Day4",
    "Day6"
]

adata_Tg_Tcells_filt.obs["day"] = pd.Categorical(
    adata_Tg_Tcells_filt.obs["day"],
    categories=day_order,
    ordered=True
)


# ------------------------------------------------------------
# 10. Cell numbers and percentages per day
# ------------------------------------------------------------

tg_day_table = (
    adata_Tg_Tcells_filt.obs
    .groupby(
        ["day", "Tg_annotation"],
        observed=False
    )
    .size()
    .rename("n_cells")
    .reset_index()
)

tg_day_table["percentage_within_day"] = (
    tg_day_table
    .groupby("day", observed=False)["n_cells"]
    .transform(lambda x: x / x.sum() * 100)
)

tg_day_table["percentage_within_day"] = (
    tg_day_table["percentage_within_day"]
    .round(1)
)

display(tg_day_table)

### 4.4 Spatial overview, cell numbers per section and day, marker genes

In [ ]:
# ============================================================
# Transgenic T-cell overview
#   1. Set annotation + colors
#   2. Spatial plot
#   3. Summary table
#   4. Barplot of cell numbers
# ============================================================


# ------------------------------------------------------------
# 1. Set transgenic annotation + colors
# ------------------------------------------------------------

adata_Tg_Tcells_filt.obs["Tg_annotation"] = (
    adata_Tg_Tcells_filt.obs["Tg_annotation"].astype("category")
)

module_palette2 = {
    "Tg_pre_Tfh": "#6baed6",
    "Tg_Tfh":     "#08519c",
    "Tg_pre_Th1": "#fdae6b",
    "Tg_Th1":     "#a63603"
}

# Explicit plotting order
tg_order = [
    "Tg_pre_Tfh",
    "Tg_Tfh",
    "Tg_pre_Th1",
    "Tg_Th1"
]

# Make sure only categories that are actually present are used
present_tg = [
    x for x in tg_order
    if x in adata_Tg_Tcells_filt.obs["Tg_annotation"].cat.categories
]

adata_Tg_Tcells_filt.obs["Tg_annotation"] = (
    adata_Tg_Tcells_filt.obs["Tg_annotation"]
    .cat.remove_unused_categories()
    .cat.reorder_categories(present_tg, ordered=True)
)

# Set Squidpy/Scanpy colors
adata_Tg_Tcells_filt.uns["Tg_annotation_colors"] = [
    module_palette2[c] for c in present_tg
]


# ------------------------------------------------------------
# 2. Spatial plot of transgenic cells
# ------------------------------------------------------------

sq.pl.spatial_scatter(
    adata_Tg_Tcells_filt,
    library_id="spatial",
    shape=None,
    color=["Tg_annotation"],
    wspace=0.4,
)


# ------------------------------------------------------------
# 3. Summary table
# ------------------------------------------------------------

tg_table = (
    adata_Tg_Tcells_filt.obs["Tg_annotation"]
    .value_counts(sort=False)
    .reindex(present_tg)
    .rename("n_cells")
    .reset_index()
)

# Rename annotation column robustly
tg_table.columns = ["Tg_annotation", "n_cells"]

# Percentage of all transgenic cells
tg_table["percentage"] = (
    tg_table["n_cells"]
    / tg_table["n_cells"].sum()
    * 100
)

tg_table["percentage"] = tg_table["percentage"].round(1)

# Display table
display(tg_table)


# ------------------------------------------------------------
# 4. Barplot of transgenic cell numbers
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.5, 4.5))

bars = ax.bar(
    tg_table["Tg_annotation"],
    tg_table["n_cells"],
    color=[
        module_palette2[x]
        for x in tg_table["Tg_annotation"]
    ],
    edgecolor="black",
    linewidth=0.6
)

# ------------------------------------------------------------
# Add n and percentage above bars
# ------------------------------------------------------------

for bar, n, pct in zip(
    bars,
    tg_table["n_cells"],
    tg_table["percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"n = {n:,}\n{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_xlabel("")
ax.set_ylabel("Number of transgenic T cells")
ax.set_title("Transgenic T-cell composition")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(
    rotation=45,
    ha="right"
)

# Give labels some room above bars
ax.set_ylim(
    0,
    tg_table["n_cells"].max() * 1.18
)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 5. Table per day and transgenic cell type
# ------------------------------------------------------------

day_col = "timepoint"

tg_day_table = (
    adata_Tg_Tcells_filt.obs
    .groupby([day_col, "Tg_annotation"], observed=False)
    .size()
    .rename("n_cells")
    .reset_index()
)

# Keep desired cell-type order
tg_day_table["Tg_annotation"] = pd.Categorical(
    tg_day_table["Tg_annotation"],
    categories=tg_order,
    ordered=True
)

tg_day_table = tg_day_table.sort_values(
    [day_col, "Tg_annotation"]
).reset_index(drop=True)

# Percentage within each day
tg_day_table["percentage_within_day"] = (
    tg_day_table["n_cells"]
    / tg_day_table.groupby(day_col)["n_cells"].transform("sum")
    * 100
).round(1)

display(tg_day_table)


# ------------------------------------------------------------
# 6. Wide count table: rows = day, columns = Tg cell type
# ------------------------------------------------------------

tg_day_counts_wide = (
    tg_day_table
    .pivot(
        index=day_col,
        columns="Tg_annotation",
        values="n_cells"
    )
    .fillna(0)
    .astype(int)
    .reindex(columns=tg_order)
)

display(tg_day_counts_wide)


# ------------------------------------------------------------
# 7. Wide percentage table
# ------------------------------------------------------------

tg_day_pct_wide = (
    tg_day_table
    .pivot(
        index=day_col,
        columns="Tg_annotation",
        values="percentage_within_day"
    )
    .fillna(0)
    .reindex(columns=tg_order)
)

display(tg_day_pct_wide)

In [ ]:
# ------------------------------------------------------------
# Summarize transgenic cell counts per biological day
# ------------------------------------------------------------

# Extract Day0, Day2, Day3, Day4, Day6 from the sample/timepoint string
tg_day_table["day"] = (
    tg_day_table["timepoint"]
    .astype(str)
    .str.extract(r"(Day\d+)", expand=False)
)

# Sum cell numbers across all samples belonging to the same day
tg_day_summary = (
    tg_day_table
    .groupby(["day", "Tg_annotation"], observed=False)["n_cells"]
    .sum()
    .reset_index()
)

# Within-day percentage after pooling samples
tg_day_summary["percentage_within_day"] = (
    tg_day_summary["n_cells"]
    / tg_day_summary.groupby("day")["n_cells"].transform("sum")
    * 100
).round(1)

# Put days in biological order
day_order = ["Day0", "Day2", "Day3", "Day4", "Day6"]

tg_day_summary["day"] = pd.Categorical(
    tg_day_summary["day"],
    categories=day_order,
    ordered=True
)

tg_day_summary["Tg_annotation"] = pd.Categorical(
    tg_day_summary["Tg_annotation"],
    categories=tg_order,
    ordered=True
)

tg_day_summary = (
    tg_day_summary
    .sort_values(["day", "Tg_annotation"])
    .reset_index(drop=True)
)

display(tg_day_summary)


# ------------------------------------------------------------
# Wide count table
# ------------------------------------------------------------

tg_day_counts_wide = (
    tg_day_summary
    .pivot(
        index="day",
        columns="Tg_annotation",
        values="n_cells"
    )
    .fillna(0)
    .astype(int)
    .reindex(index=day_order, columns=tg_order)
)

display(tg_day_counts_wide)


# ------------------------------------------------------------
# Wide percentage table
# ------------------------------------------------------------

tg_day_pct_wide = (
    tg_day_summary
    .pivot(
        index="day",
        columns="Tg_annotation",
        values="percentage_within_day"
    )
    .fillna(0)
    .reindex(index=day_order, columns=tg_order)
)

display(tg_day_pct_wide)

In [ ]:
sc.tl.rank_genes_groups(
    adata_Tg_Tcells_filt,
    groupby="Tg_annotation",
    method="wilcoxon",     # or 't-test' / 'logreg'
    n_genes=adata_Tg_Tcells.raw.shape[1] if adata_Tg_Tcells.raw is not None else None
)

In [ ]:
sc.pl.rank_genes_groups(adata_Tg_Tcells_filt, n_genes=50, sharey=False)

## 5. Refined annotation of all cells

Extended marker set; GC B cells = top 25 % Aicda score; only QC- and score-filtered Tg cells are kept
as `Tg_T_cells`, all other Tg calls and unassigned cells are removed.

In [ ]:
ln_major_markers2  = {
    "Tregs": ["Cd3d","Cd4", "Foxp3", "Il2ra"],
    "cDC1": ["Xcr1", "Itgax", "Cd74", "Irf8"],
    "cDC2": ["Sirpa", "Itgax", "Cd74", "Il6"],
    #"GC_B_cells": ["Cd79a", "Cxcr5", "Aicda", "Bcl6"],
    "GC_B_cells": ["Aicda"],
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"],
    "Neutrophils" :["Lst1", "Csf3r", "Pglyrp1", "Mmp9"]
}

In [ ]:
# 1. Compute gene set scores
for comp, genes in ln_major_markers2.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{comp}_score")

# 2. Get compartment list
compartments = list(ln_major_markers2.keys())

# 3. Annotate each cell by its highest score
adata.obs["compartment_annotation_2"] = adata.obs[
    [f"{c}_score" for c in compartments]
].idxmax(axis=1).str.replace("_score", "")

In [ ]:
adata.obs["compartment_annotation_2"].value_counts()

### GC B cells – keep top 25 % Aicda score

In [ ]:
adata_gc = adata[adata.obs['compartment_annotation_2'] == 'GC_B_cells'].copy()

In [ ]:
plt.figure(figsize=(2, 5))
sns.boxplot(
    data=adata_gc.obs,
    x="compartment_annotation_2",
    y="GC_B_cells_score",
    showcaps=True,
    fliersize=1,
    linewidth=1,
    color="skyblue"
)

plt.xticks(rotation=45, ha="right")
plt.title("Distribution of GC_B_cells module scores")
plt.tight_layout()


plt.show()

In [ ]:
print("GC_B_cells_score stats:")
print(adata_gc.obs['GC_B_cells_score'].describe())
print("\nTop 25% threshold:", adata_gc.obs['GC_B_cells_score'].quantile(0.75))
print("Max score:", adata_gc.obs['GC_B_cells_score'].max())

In [ ]:
#subset GC_B_cells to only HIGH Aicda scores
high_gc_threshold = adata_gc.obs['GC_B_cells_score'].quantile(0.75) # still high for 1.55
gc_mask = (adata.obs['compartment_annotation_2'] == 'GC_B_cells') & \
          (adata.obs['GC_B_cells_score'] > high_gc_threshold)

# Reassign low-confidence GC_B_cells
adata.obs.loc[~gc_mask & (adata.obs['compartment_annotation_2'] == 'GC_B_cells'),
              'compartment_annotation_2'] = 'B_cells' #(for the time being will change later)

print(f"High-confidence GC_B_cells: {gc_mask.sum()}")
print(adata.obs['compartment_annotation_2'].value_counts())

### Keep only filtered Tg cells, remove unassigned cells

In [ ]:
adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].astype(str)

# 1. filtered Tg cells
filtered_cells = set(adata_Tg_Tcells_filt.obs_names)

# 2. all cells annotated as Tg
is_tg_original = adata.obs["compartment_annotation_2"].str.startswith("Tg")

# 3. set them to NA
adata.obs.loc[is_tg_original, "compartment_annotation_2"] = "NA"

# 4. restore the filtered Tg cells
adata.obs.loc[adata.obs_names.isin(filtered_cells), "compartment_annotation_2"] = "Tg_T_cells"

# 5. remove NA
adata = adata[adata.obs["compartment_annotation_2"] != "NA"].copy()
adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].astype("category")

print((adata.obs["compartment_annotation_2"] == "Tg_T_cells").sum(), "Tg cells")  # should equal len(filtered_cells)

In [ ]:
adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].astype("category")
adata.uns["compartment_annotation_colors"] = [
    ln_major_colors[c] for c in adata.obs["compartment_annotation_2"].cat.categories
]
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "compartment_annotation_2",
    ],
    wspace=0.4,
)

### Composition per section (final annotation)

In [ ]:
# Remove entries with "Unknown" timepoint
adata_filtered = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata_filtered.obs.groupby(["timepoint", "compartment_annotation_2"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation_2', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Export

### Save all annotated cells (input for composition analyses)

In [ ]:
adata.write(ALL_OUT)
print(adata.shape, '->', ALL_OUT)

### Tg + T + B cells (input for T/B-border inference)

In [ ]:
# pick your refined label column in adata_Tg_Tcells_filt
refined_key = "Tg_annotation"

# create a column in full adata with refined Tg labels (others = "non_Tg")
adata.obs["Tg_refined"] = "non_Tg"
adata.obs.loc[adata_Tg_Tcells_filt.obs_names, "Tg_refined"] = (
    adata_Tg_Tcells_filt.obs[refined_key].astype(str).values
)
adata.obs["Tg_refined"] = adata.obs["Tg_refined"].astype("category")

In [ ]:
adata_TB = adata[adata.obs["compartment_annotation_2"].isin(["B_cells", "T_cells"])].copy()
print(adata_TB.shape)

adata_TgT_B = ad.concat(
    [adata_Tg_Tcells_filt, adata_TB],
    join="outer",
    label="source",
    keys=["Tg_cells", "TB_cells"],
    fill_value=0
)
print(adata_TgT_B.shape)


adata_TgT_B.obs["Tg_annotation_full"] = np.select(
    [
        adata_TgT_B.obs["compartment_annotation_2"] == "B_cells",
        adata_TgT_B.obs["compartment_annotation_2"] == "T_cells"
    ],
    [
        "B_cells",
        "T_cells"
    ],
    default=adata_TgT_B.obs["Tg_annotation"]
)

# Make it categorical
adata_TgT_B.obs["Tg_annotation_full"] = adata_TgT_B.obs["Tg_annotation_full"].astype("category")

# Quick sanity check
adata_TgT_B.obs["Tg_annotation_full"].value_counts()

In [ ]:
adata_TgT_B.write(TB_OUT)
print(adata_TgT_B.shape, '->', TB_OUT)